# Pipeline and results

## What the pipeline does

The entry point is run.py at the repository root: `python run.py`. It runs one clip from input to output:

1. **Preflight.** Config, manifest, dataset directory and input clip must exist. A missing input aborts with exit code 2.
2. **Verify.** The clip's size and SHA-256 are checked against the manifest. A mismatch aborts before any processing, exit code 3.
3. **Load.** The clip is loaded and must satisfy the audio contract: int16, shape (frames, channels), consistent with the file header. A violation aborts with exit code 4.
4. **Encode.** APAC with the quant_step from config.json (1 = lossless).
5. **Decode.**
6. **Check.** A lossless run must reconstruct the input bit-exactly, otherwise exit code 5.
7. **Save.** compressed.apac, decoded.wav and results.json, written atomically: every file is completed under a temp name and renamed into place, so an interrupted run (exit code 130) leaves no partial files.

Every step appends one line to output/run.log. The pipeline has no random steps: the same input, config and codec version produce the same output bytes, so no seed is needed.

The cell below runs the entry point as shipped, with the default configuration (a fold1 clip inside p2/data/raw).

In [1]:
import subprocess
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "run.py").is_file():
    root = root.parent

run = subprocess.run([sys.executable, "run.py"], capture_output=True, text=True, cwd=root)
print(run.stdout)
print("exit code:", run.returncode)


run started: input fold1/124489-9-0-12.wav, quant_step 1
verify: sha256 ok for fold1/124489-9-0-12.wav
load: fold1/124489-9-0-12.wav frames=176400 channels=2 rate=44100 subtype=PCM_16
encode: 476353 bytes in 0.086s
decode: 176400 frames in 0.692s
save: compressed.apac, decoded.wav, results.json -> output
run finished: bit_exact=True ratio=1.481 in 0.79s
input_file: fold1/124489-9-0-12.wav
input_sha256: adab612f876afd5f13d2ef8d39d64a466507c77d6d00aeee3f880542b6386946
sample_rate: 44100
frames: 176400
channels: 2
subtype: PCM_16
quant_step: 1
wav_bytes: 705644
apac_bytes: 476353
compression_ratio: 1.481
space_saving_percent: 32.49
bit_exact: True
rmse: 0.0
encode_seconds: 0.0857
decode_seconds: 0.692
encode_peak_bytes: 2823528
decode_peak_bytes: 16843228
run_seconds: 0.791
timestamp_utc: 2026-11-03T13:41:57+00:00
environment: {'python': '3.13.7', 'numpy': '2.3.3', 'soundfile': '0.13.1'}

exit code: 0


## Results

The same metrics are saved as output/results.json by the run above.

In [2]:
import json

results = json.loads((root / "output" / "results.json").read_text(encoding="utf-8"))
for key, value in results.items():
    print(f"{key}: {value}")


input_file: fold1/124489-9-0-12.wav
input_sha256: adab612f876afd5f13d2ef8d39d64a466507c77d6d00aeee3f880542b6386946
sample_rate: 44100
frames: 176400
channels: 2
subtype: PCM_16
quant_step: 1
wav_bytes: 705644
apac_bytes: 476353
compression_ratio: 1.481
space_saving_percent: 32.49
bit_exact: True
rmse: 0.0
encode_seconds: 0.0857
decode_seconds: 0.692
encode_peak_bytes: 2823528
decode_peak_bytes: 16843228
run_seconds: 0.791
timestamp_utc: 2026-11-03T13:41:57+00:00
environment: {'python': '3.13.7', 'numpy': '2.3.3', 'soundfile': '0.13.1'}


## Timing and memory evidence

One storage choice, measured on the same clip: keep the PCM WAV file, or store it as APAC at quantization step 1 (lossless) or step 4 (lossy; the first step past the measured FLAC crossover point in the p2 evaluation). Wall time comes from time.perf_counter, memory from the tracemalloc peak around each stage.

The numbers come from one run on one machine. The method is the evidence; the recorded values are an example of it.

In [3]:
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(root / "src"))
from p3 import config as cfg, pipeline

config = cfg.load_config(root)
bench = pipeline.benchmark(root, config)
print("input:", bench["input_file"])
display(pd.DataFrame(bench["rows"]))


input: fold1/124489-9-0-12.wav


,storage,bytes,encode_s,decode_s,encode_peak_mb,decode_peak_mb,bit_exact,rmse
0,WAV PCM_16 (original file),705644,NaN,NaN,NaN,NaN,True,0.0000
1,APAC quant_step 1,476353,0.6285,5.2573,2.69,16.06,True,0.0000
2,APAC quant_step 4,387775,18.4929,5.9226,2.69,15.97,False,0.8661


## Failure handling

The four required failure cases are covered by tests, one fixture each:

| Case | What happens | Where |
|---|---|---|
| Missing input | preflight names the missing path, exit code 2, nothing runs | test_missing_input_aborts_before_anything_runs |
| Digest mismatch | verification aborts before encoding, exit code 3, no outputs | test_digest_mismatch_aborts_before_encoding |
| Invalid shape | the contract rejects 1D, non-int16 and header-mismatched arrays, exit code 4 | test_invalid_shape_is_rejected |
| Interrupted run | temp files are removed, nothing partial is kept, exit code 130, the log records the step | test_interrupted_run_leaves_no_partial_outputs |

The cell below runs the test suite; the transcript is the evidence that all cases pass.

In [4]:
tests = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"],
                       capture_output=True, text=True, cwd=root)
print(tests.stdout)
print("exit code:", tests.returncode)


.........                                                                [100%]
9 passed in 1.11s

exit code: 0


## Where it stops being reliable

- **Coverage.** The repository carries fold1 (873 of 8,732 excerpts) inside p2/data/raw; the default pipeline run uses one clip from it. The manifest covers the full release, and `python run.py --verify-only --dataset-dir PATH` verifies any complete copy against it.
- **Measurements.** Timing and memory numbers vary between machines and runs. The output bytes do not: a lossless run is bit-exact, which run.py asserts on every run.
- **Labels.** No check verifies that the class labels describe the audio correctly; the pipeline does not use the labels.
- **Loading.** The codec's working format is int16; wider-than-16-bit and float sources carry at most 1/2 LSB of quantization error per sample after loading. This is a property of the pipeline, not of the dataset.
- **Licence.** CC BY-NC 4.0 (per the P1 audit) restricts commercial use of the data.